# NOVA SN68 — scoring BoltzGen de nanobodies sur GPU Kaggle

Note des séquences avec **la version de BoltzGen embarquée par les validateurs NOVA**, avec leur structure cible (Q9NZQ7 / 8AOK), leur site de liaison, leur protocole et leurs échelles.

Notebook **autonome** : la source BoltzGen, la structure, le lot (`inputs/kaggle_nanobody_batch.csv`) et le code de scoring sont intégrés. La MSA est téléchargée via jsDelivr et vérifiée par empreinte SHA-256 ; les poids (~6 Go) viennent de Hugging Face.

**Réglages Kaggle :** *Accelerator* **GPU T4 x2**, *Internet* **On**.
- Test de durée : `TOP = 2` puis **Run All** dans l'éditeur.
- Lot complet : `TOP = None`, puis **Save Version → Save & Run All (Commit)**.

Les noyaux cuEquivariance se désactivent d'eux-mêmes sur T4 (capacité 7.5 < 8), BoltzGen prend son chemin de repli. **Durée par séquence inconnue** : c'est ce que ce notebook doit mesurer.

In [ ]:
# Paramètres
TOP = 2             # None = tout le lot
RUN = "nanobody_batch1"

import base64, io, os, pathlib, subprocess, sys, tarfile, time
T0 = time.time()
TEMP = pathlib.Path("/kaggle/temp")
WORK = TEMP / "Nova"
NOVA = TEMP / "nova"
VENV = TEMP / "bgvenv"
OUT = pathlib.Path("/kaggle/working")

def step(msg):
    print(f"[{time.strftime('%H:%M:%S')}] +{(time.time() - T0) / 60:5.1f} min  {msg}", flush=True)

def sh(cmd, **kw):
    res = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)
    out = (res.stdout + res.stderr).strip()
    if out:
        print(out[-3000:])
    if res.returncode != 0:
        raise RuntimeError(f"échec ({res.returncode}) : {cmd}")
    return res.stdout

step("démarrage")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
sh("df -h /kaggle/temp | tail -1")

In [ ]:
# Données intégrées : BoltzGen des validateurs, structure 8AOK, inputs/kaggle_nanobody_batch.csv, code de scoring (tar.xz base64)
MSA_NAME = "Q9NZQ7.a3m"
MSA_SHA256 = "0bd1fc43143c7881b52a35cc42903eb3ea37c4791e17f5f3837c20b59b5101c0"
MSA_URLS = ["https://cdn.jsdelivr.net/gh/metanova-labs/nova@db9bf190a6907338a05871b9658f42a089229675/data/msa_files/Q9NZQ7.a3m", "https://raw.githubusercontent.com/metanova-labs/nova/db9bf190a6907338a05871b9658f42a089229675/data/msa_files/Q9NZQ7.a3m"]
PAYLOAD = "..."

In [ ]:
step("test de l'accès Internet")
for url in ["https://huggingface.co", "https://pypi.org", "https://cdn.jsdelivr.net"]:
    rc = subprocess.run(["curl", "-s", "-o", "/dev/null", "-m", "15", "-w", "%{http_code}", url],
                        capture_output=True, text=True).stdout or "000"
    print(f"  {url}: {rc}")
    if rc == "000":
        raise SystemExit("Pas d'accès Internet : activer 'Internet' dans les paramètres du notebook.")

In [ ]:
step("extraction du code, de la source BoltzGen et de la structure")
for d in (WORK, OUT):
    d.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:xz") as tar:
    tar.extractall(WORK)
(TEMP / "nova").mkdir(parents=True, exist_ok=True)
sh(f"cp -r {WORK}/nova/. {NOVA}/ && rm -rf {WORK}/nova")
step("téléchargement de la MSA des validateurs")
import hashlib
msa = NOVA / "data/msa_files" / MSA_NAME
msa.parent.mkdir(parents=True, exist_ok=True)
for url in MSA_URLS:
    subprocess.run(["curl", "-sSL", "-m", "180", "--retry", "2", "-o", str(msa), url])
    if msa.exists() and hashlib.sha256(msa.read_bytes()).hexdigest() == MSA_SHA256:
        print(f"  MSA OK depuis {url}")
        break
    print(f"  échec depuis {url}")
else:
    raise RuntimeError("MSA introuvable ou corrompue")
sh(f"find {NOVA} -maxdepth 3 -type d | sort && ls {WORK}")
step("étape 1 terminée")

## Environnement BoltzGen
La version des validateurs épingle `numpy==2.0.2`, `torch==2.7.1` et une trentaine d'autres paquets : elle est installée dans un venv séparé (compter ~10 min).

In [ ]:
step("installation de uv + Python 3.12")
sh(f"{sys.executable} -m pip install -q uv")
sh(f"uv venv -q --python 3.12 {VENV}")
py = str(VENV / "bin/python")
step("installation de torch 2.7.1 CUDA")
sh(f"uv pip install -q --python {py} torch==2.7.1 --index-url https://download.pytorch.org/whl/cu126")
step("installation de la BoltzGen des validateurs (+ pandas, requests)")
sh(f"uv pip install -q --python {py} {NOVA}/boltzgen pandas requests")
sh(f'{py} -c "import torch, boltzgen; print(torch.__version__, torch.cuda.is_available(), '
   f'torch.cuda.get_device_capability(0))"')
step("étape 2 terminée")

## Scoring
Le premier lancement télécharge les poids BoltzGen (~6 Go). La progression s'affiche toutes les 5 minutes avec la dernière ligne du journal ; le journal complet est dans `/kaggle/working/boltzgen_<RUN>.log`.

In [ ]:
import pandas as pd
BATCH = "inputs/kaggle_nanobody_batch.csv"
os.chdir(WORK)
n_total = len(pd.read_csv(BATCH).head(TOP) if TOP else pd.read_csv(BATCH))
log_path = OUT / f"boltzgen_{RUN}.log"
cmd = [py, "-m", "scripts.boltzgen_score", "--file", BATCH, "--run", RUN,
       "--nova-dir", str(NOVA), "--boltzgen", str(VENV / "bin/boltzgen")]
if TOP:
    cmd += ["--top", str(TOP)]

def last_log_line():
    lines = [l.strip() for l in log_path.read_text(errors="replace").replace("\r", "\n").splitlines() if l.strip()]
    return lines[-1][:200] if lines else "(journal vide)"

step(f"lancement de BoltzGen sur {n_total} séquences")
with open(log_path, "w") as log:
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    while proc.poll() is None:
        time.sleep(300)
        step(f"en cours — {last_log_line()}")

print(log_path.read_text(errors="replace").replace("\r", "\n")[-5000:])
step(f"BoltzGen terminé (code {proc.returncode})")
assert proc.returncode == 0, "BoltzGen a échoué : voir le journal ci-dessus"

In [ ]:
import shutil
src = WORK / f"data/boltzgen/{RUN}/scores.csv"
out = OUT / f"scores_nanobody_{RUN}.csv"
shutil.copy(src, out)
df = pd.read_csv(out)
step(f"résultats : {out} ({len(df)} séquences)")
df